# Combined `PVPortfolio` (PV_Base + PV_LC_NR)

Build one portfolio from:

- **30** `PresentValueInstrument`s — Input 4 / `PV_Base` (USD)
- **3** `LocalCurrencyNonResidentInstrument`s — Input 5 / `PV_LC_NR*` (LC → USD)

Both expose the same canonical Output rows, so they share a `PVPortfolio`.

See `docs/03-pv-instruments.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

from lic_dsf.load import (
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
)
from lic_dsf.pv import (
    LocalCurrencyNonResidentInstrument,
    PresentValueInstrument,
    PVPortfolio,
)

pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
WORKBOOK

PosixPath('/home/sravan/excel-grapher/lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

## Load PV_Base instruments (`PresentValueInstrument`)

Input 4 lines with complete terms → 30 instruments (PC2–PC5 skip when grace/maturity are empty).

In [2]:
pv_base = load_instruments_from_workbook(WORKBOOK)
assert all(isinstance(i, PresentValueInstrument) for i in pv_base)

pv_catalog = pd.DataFrame(
    [
        {
            "name": i.name,
            "class": type(i).__name__,
            "grace": i.grace,
            "maturity": i.maturity,
            "interest": i.interest_rate,
            "discount": i.discount_rate,
            "disbursement_sum": sum(i.disbursements),
        }
        for i in pv_base
    ]
)
print(f"{len(pv_base)} PresentValueInstrument(s)")
pv_catalog

30 PresentValueInstrument(s)


,name,class,grace,maturity,interest,discount,disbursement_sum
0,IMF,PresentValueInstrument,5,10,0.0025,0.0500,0.0000
1,IDA - regular,PresentValueInstrument,6,38,0.0075,0.0500,790.3180
2,IDA - 50Y loans,PresentValueInstrument,10,50,0.0000,0.0500,300.0000
3,IDA - SML,PresentValueInstrument,6,12,0.0000,0.0500,10.0000
4,IDA NEW 40-year credits,PresentValueInstrument,11,40,0.0000,0.0500,10.0000
5,IDA NEW Regular,PresentValueInstrument,6,31,0.0075,0.0500,100.0000
6,IDA NEW Blend (also enter) -->,PresentValueInstrument,5,25,0.0324,0.0500,120.0000
7,IDA NEW 60-year credits,PresentValueInstrument,20,60,0.0000,0.0500,120.0000
8,MULTI1,PresentValueInstrument,5,30,0.0075,0.0500,0.0000
9,MULTI2,PresentValueInstrument,5,30,0.0000,0.0500,0.0000


## Load PV_LC_NR instruments (`LocalCurrencyNonResidentInstrument`)

Three LC non-resident bond tenors from Input 5 + Macro-Debt FX.

In [3]:
lc_nr = load_lc_nr_instruments_from_workbook(WORKBOOK)
assert all(isinstance(i, LocalCurrencyNonResidentInstrument) for i in lc_nr)

lc_catalog = pd.DataFrame(
    [
        {
            "name": i.name,
            "class": type(i).__name__,
            "grace": i.grace,
            "maturity": i.maturity,
            "discount": i.discount_rate,
            "disbursement_lc_sum": sum(i.disbursements_lc),
        }
        for i in lc_nr
    ]
)
print(f"{len(lc_nr)} LocalCurrencyNonResidentInstrument(s)")
lc_catalog

3 LocalCurrencyNonResidentInstrument(s)


,name,class,grace,maturity,discount,disbursement_lc_sum
0,Bonds (1 to 3 years)-LC,LocalCurrencyNonResidentInstrument,1,2,0.0500,"154,963.8038"
1,Bonds (4 to 7 years)-LC,LocalCurrencyNonResidentInstrument,3,5,0.0500,"130,428.3091"
2,Bonds (beyond 7 years)-LC,LocalCurrencyNonResidentInstrument,6,7,0.0500,"256,618.2082"


## Combined `PVPortfolio`

Concatenate USD PV_Base lines with LC-NR tenors (33 instruments total).

In [4]:
instruments = tuple(pv_base) + tuple(lc_nr)
portfolio = PVPortfolio(instruments)

summary = pd.DataFrame(
    [
        {
            "name": i.name,
            "class": type(i).__name__,
        }
        for i in portfolio.instruments
    ]
)
print(f"{len(portfolio.instruments)} instruments in portfolio")
print(summary["class"].value_counts().to_string())
summary

33 instruments in portfolio
class
PresentValueInstrument                30
LocalCurrencyNonResidentInstrument     3


,name,class
0,IMF,PresentValueInstrument
1,IDA - regular,PresentValueInstrument
2,IDA - 50Y loans,PresentValueInstrument
3,IDA - SML,PresentValueInstrument
4,IDA NEW 40-year credits,PresentValueInstrument
5,IDA NEW Regular,PresentValueInstrument
6,IDA NEW Blend (also enter) -->,PresentValueInstrument
7,IDA NEW 60-year credits,PresentValueInstrument
8,MULTI1,PresentValueInstrument
9,MULTI2,PresentValueInstrument


## Portfolio metrics

Per-instrument Interest / Amortization / PV / Stock, plus aggregated new debt service.

In [5]:
portfolio.interest().iloc[:, :8]

,2024,2025,2026,2027,2028,2029,2030,2031
IMF,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
IDA - regular,0.0000,1.1250,1.2750,2.1352,2.9274,4.4274,5.9274,5.9274
IDA - 50Y loans,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
IDA - SML,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
IDA NEW 40-year credits,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
IDA NEW Regular,0.0000,0.0000,0.0000,0.7500,0.7500,0.7500,0.7500,0.7500
IDA NEW Blend (also enter) -->,0.0000,0.0000,0.6480,1.9440,3.8880,3.8880,3.8880,3.8880
IDA NEW 60-year credits,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
MULTI1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
MULTI2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [6]:
portfolio.aggregate_external().iloc[:, :10]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
"New forex borrowing (gross, USD)","4,413.5528","3,381.2959","3,032.4600","3,645.9940","3,986.2384","4,137.4496","4,394.7924","4,306.5683","4,730.9868","4,952.6171"
cumulative,"4,413.5528","7,794.8487","10,827.3087","14,473.3026","18,459.5410","22,596.9907","26,991.7830","31,298.3513","36,029.3382","40,981.9552"
Stock of new forex debt (in USD),"4,234.7734","7,268.4932","9,545.0717","12,101.6633","14,463.6845","16,785.5115","19,415.6333","20,471.1065","22,364.2226","24,647.4380"
PV of debt,"4,061.9963","6,833.1360","8,725.8791","10,910.2984","13,040.5630","15,219.9064","17,833.6104","18,818.5881","20,653.5132","22,890.8988"
Total debt service (in USD),0.0000,492.2689,"1,172.3549","1,718.3883","2,420.0663","2,677.3321","2,681.4832","4,318.5720","3,852.9187","3,719.7739"
Interest,0.0000,492.2689,796.4728,960.2668,"1,098.9568","1,187.2116","1,262.3852","1,405.5394","1,336.3602","1,376.3506"
Amortization,0.0000,0.0000,375.8820,758.1215,"1,321.1095","1,490.1205","1,419.0980","2,913.0325","2,516.5585","2,343.4233"


In [7]:
portfolio.new_debt_service().iloc[:, :10]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
Interest,0.0000,492.2689,796.4728,960.2668,"1,098.9568","1,187.2116","1,262.3852","1,405.5394","1,336.3602","1,376.3506"
Amortization,0.0000,0.0000,375.8820,758.1215,"1,321.1095","1,490.1205","1,419.0980","2,913.0325","2,516.5585","2,343.4233"
Total new debt service,0.0000,492.2689,"1,172.3549","1,718.3883","2,420.0663","2,677.3321","2,681.4832","4,318.5720","3,852.9187","3,719.7739"
